In [6]:
import gc
import json
import os
import pickle
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch as t
from datasets import load_dataset
from dotenv import load_dotenv
from IPython.display import HTML, display
from jaxtyping import Bool, Float
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from torch import Tensor
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

device = t.device("cuda" if t.cuda.is_available() else "cpu")
dtype = t.bfloat16

In [7]:
import os
from pathlib import Path

# Ensure cwd is the repo root regardless of where the kernel started
repo_root = Path(__file__).parent.parent if "__file__" in dir() else Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
print(f"Working directory: {Path.cwd()}")

Working directory: /workspace/SPAR-causal-probes


In [8]:
load_dotenv(dotenv_path=str(".env"))
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "Please set HF_TOKEN in your chapter1_transformer_interp/exercises/.env file"

In [9]:
import pandas as pd

df = pd.read_json("dct_probes/results/judge_results.jsonl", lines=True)

baseline = df[df["factor_idx"] == -1].set_index("prompt_id")["judge_score"]
print("Baseline scores:")
print(baseline)
print(f"\nMean baseline truthfulness: {baseline.mean():.1f}")

steered = df[df["factor_idx"] >= 0].copy()
steered = steered.merge(baseline.rename("baseline_score"), on="prompt_id")
steered["delta"] = steered["baseline_score"] - steered["judge_score"]

mean_deltas = steered.groupby("factor_idx")["delta"].mean().sort_values(ascending=False)
print(f"\nTop 10 truth-reducing vectors:")
print(mean_deltas.head(10))
print(f"\nTop 10 truth-increasing vectors:")
print(mean_deltas.tail(10))

Baseline scores:
prompt_id
factual_geography       10
factual_science         10
factual_history         10
numerical_reasoning      0
common_misconception    10
ambiguous_ethics        -1
self_knowledge           8
misleading_premise       0
statistical_claim       10
causal_reasoning         2
Name: judge_score, dtype: int64

Mean baseline truthfulness: 5.9

Top 10 truth-reducing vectors:
factor_idx
57     6.2
31     6.1
45     6.1
83     6.1
99     6.1
105    6.1
15     6.0
0      6.0
28     6.0
25     6.0
Name: delta, dtype: float64

Top 10 truth-increasing vectors:
factor_idx
36     5.4
32     5.4
119    5.3
66     5.0
30     4.9
60     4.9
118    4.9
78     4.9
93     3.9
29     3.5
Name: delta, dtype: float64


# Loading the model

In [10]:
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto",
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

NUM_LAYERS = len(model.model.layers)
D_MODEL = model.config.hidden_size
# Layer choices matching dct_params.json for Llama-3.2-1B-Instruct
PROBE_LAYER = 14
INTERVENE_LAYER = 4

print(f"Model: {MODEL_NAME}")
print(f"Layers: {NUM_LAYERS}, Hidden dim: {D_MODEL}")
print(f"Probe layer: {PROBE_LAYER}, Intervene layer: {INTERVENE_LAYER}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model: meta-llama/Llama-3.1-8B-Instruct
Layers: 32, Hidden dim: 4096
Probe layer: 14, Intervene layer: 4


# Loading the datasets

In [11]:
DATASET_NAMES = ["cities", "sp_en_trans", "larger_than"]

input_dir = Path("dct_probes/data")

datasets = {}
for name in DATASET_NAMES:
    df = pd.read_csv(input_dir / f"got_datasets" / f"{name}.csv")
    datasets[name] = df
    print(f"\n{name}: {len(df)} statements ({df['label'].sum()} true, {(1 - df['label']).sum():.0f} false)")
    display(df.head(4))


cities: 1496 statements (748 true, 748 false)


,statement,label,city,country,correct_country
0,The city of Krasnodar is in Russia.,1,Krasnodar,Russia,Russia
1,The city of Krasnodar is in South Africa.,0,Krasnodar,South Africa,Russia
2,The city of Lodz is in Poland.,1,Lodz,Poland,Poland
3,The city of Lodz is in the Dominican Republic.,0,Lodz,the Dominican Republic,Poland



sp_en_trans: 354 statements (177 true, 177 false)


,statement,label
0,The Spanish word 'con' means 'to speak'.,0
1,The Spanish word 'uno' means 'one'.,1
2,The Spanish word 'tener' means 'to have'.,1
3,The Spanish word 'caliente' means 'hot'.,1



larger_than: 1980 statements (990 true, 990 false)


,statement,label,n1,n2,diff,abs_diff
0,Fifty-one is larger than fifty-two.,0,51,52,-1,1
1,Fifty-one is larger than fifty-three.,0,51,53,-2,2
2,Fifty-one is larger than fifty-four.,0,51,54,-3,3
3,Fifty-one is larger than fifty-five.,0,51,55,-4,4


# Extract activations

In [12]:
def extract_activations(
    statements: list[str],
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    layers: list[int],
    batch_size: int = 25,
) -> dict[int, Float[Tensor, "n_statements d_model"]]:
    """
    Extract last-token hidden state activations from specified layers for a list of statements.

    Args:
        statements: List of text statements to process.
        model: A HuggingFace causal language model.
        tokenizer: The corresponding tokenizer.
        layers: List of layer indices (0-indexed) to extract activations from.
        batch_size: Number of statements to process at once.

    Returns:
        Dictionary mapping layer index to tensor of activations, shape [n_statements, d_model].
    """
    all_acts = {layer: [] for layer in layers}

    for i in range(0, len(statements), batch_size):
        batch = statements[i : i + batch_size]

        # Sanity check: every statement should end with a period, since the GoT paper probes
        # at the end-of-sentence punctuation token
        for stmt in batch:
            assert stmt.rstrip().endswith("."), f"Statement doesn't end with period: {stmt!r}"

        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)

        with t.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        # Find the last non-padding token index for each sequence
        last_token_idx = inputs["attention_mask"].sum(dim=1) - 1  # [batch]

        for layer in layers:
            # hidden_states[0] is embedding, hidden_states[layer+1] is output of layer
            hidden = outputs.hidden_states[layer + 1]  # [batch, seq_len, d_model]
            # Extract last real token for each sequence
            batch_indices = t.arange(hidden.shape[0], device=hidden.device)
            acts = hidden[batch_indices, last_token_idx]  # [batch, d_model]
            all_acts[layer].append(acts.cpu().float())

    return {layer: t.cat(acts_list, dim=0) for layer, acts_list in all_acts.items()}

In [13]:
# Extract activations at the probe layer for all datasets
activations = {}
labels_dict = {}

for name in DATASET_NAMES:
    df = datasets[name]
    statements = df["statement"].tolist()
    labs = t.tensor(df["label"].values, dtype=t.float32)

    acts = extract_activations(statements, model, tokenizer, [PROBE_LAYER])
    activations[name] = acts[PROBE_LAYER]
    labels_dict[name] = labs

# Show summary table
summary = pd.DataFrame(
    {
        "Dataset": DATASET_NAMES,
        "N statements": [len(datasets[n]) for n in DATASET_NAMES],
        "N true": [int(datasets[n]["label"].sum()) for n in DATASET_NAMES],
        "N false": [int((1 - datasets[n]["label"]).sum()) for n in DATASET_NAMES],
        "Act shape": [str(tuple(activations[n].shape)) for n in DATASET_NAMES],
        "Mean norm": [f"{activations[n].norm(dim=-1).mean():.1f}" for n in DATASET_NAMES],
    }
)
display(summary)

,Dataset,N statements,N true,N false,Act shape,Mean norm
0,cities,1496,748,748,"(1496, 4096)",9.7
1,sp_en_trans,354,177,177,"(354, 4096)",9.9
2,larger_than,1980,990,990,"(1980, 4096)",9.1


# Get PCA components

In [14]:
def get_pca_components(
    activations: Float[Tensor, "n d_model"],
    k: int = 2,
) -> Float[Tensor, "d_model k"]:
    """
    Compute the top-k principal components of the activation matrix.

    Args:
        activations: Activation matrix, shape [n_samples, d_model].
        k: Number of principal components to return.

    Returns:
        Matrix of top-k eigenvectors as columns, shape [d_model, k].
    """
    # Mean-center the data
    X = activations - activations.mean(dim=0)

    # Compute covariance matrix
    cov = X.t() @ X / (X.shape[0] - 1)

    # Eigendecompose
    eigenvalues, eigenvectors = t.linalg.eigh(cov)

    # Sort by eigenvalue descending and take top-k
    sorted_indices = t.argsort(eigenvalues, descending=True)
    top_k = eigenvectors[:, sorted_indices[:k]]

    return top_k

In [15]:
fig = make_subplots(rows=1, cols=3, subplot_titles=DATASET_NAMES)

for i, name in enumerate(DATASET_NAMES):
    acts = activations[name]
    labs = labels_dict[name]
    pcs = get_pca_components(acts, k=2)
    X_centered = acts - acts.mean(dim=0)
    projected = (X_centered @ pcs).numpy()

    # Compute variance explained
    total_var = X_centered.var(dim=0).sum().item()
    pc_var = t.tensor(projected).var(dim=0)
    pct_explained = (pc_var / total_var * 100).tolist()

    colors = ["blue" if l == 1 else "red" for l in labs.tolist()]
    fig.add_trace(
        go.Scatter(
            x=projected[:, 0],
            y=projected[:, 1],
            mode="markers",
            marker=dict(color=colors, size=3, opacity=0.5),
            name=name,
            showlegend=False,
        ),
        row=1,
        col=i + 1,
    )
    fig.update_xaxes(title_text=f"PC1 ({pct_explained[0]:.1f}%)", row=1, col=i + 1)
    fig.update_yaxes(title_text=f"PC2 ({pct_explained[1]:.1f}%)", row=1, col=i + 1)

# Add a legend manually
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(color="blue", size=8), name="True"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(color="red", size=8), name="False"))

fig.update_layout(
    title="PCA of Truth Representations (Layer 14, Last Token)",
    height=400,
    width=1200,
)
fig.show()

# Layer Sweep

In [16]:
def layer_sweep_accuracy(
    statements: list[str],
    labels: Float[Tensor, " n"],
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    layers: list[int],
    train_frac: float = 0.8,
    batch_size: int = 25,
) -> dict[str, list[float]]:
    """
    For each layer, train a difference-of-means classifier and compute train/test accuracy.

    Args:
        statements: List of statements.
        labels: Binary labels (1=true, 0=false).
        model: The language model.
        tokenizer: The tokenizer.
        layers: List of layer indices to sweep over.
        train_frac: Fraction of data for training.
        batch_size: Batch size for activation extraction.

    Returns:
        Dict with keys "train_acc" and "test_acc", each a list of accuracies per layer.
    """
    # Split into train/test
    n_train = int(len(statements) * train_frac)
    perm = t.randperm(len(statements))
    train_idx, test_idx = perm[:n_train], perm[n_train:]
    train_statements = [statements[i] for i in train_idx]
    test_statements = [statements[i] for i in test_idx]
    train_labels = labels[train_idx]
    test_labels = labels[test_idx]

    # Extract activations at all layers at once
    train_acts = extract_activations(train_statements, model, tokenizer, layers, batch_size)
    test_acts = extract_activations(test_statements, model, tokenizer, layers, batch_size)

    train_accs = []
    test_accs = []

    for layer in layers:
        tr_acts = train_acts[layer]
        te_acts = test_acts[layer]

        # Difference of means direction
        true_mean = tr_acts[train_labels == 1].mean(dim=0)
        false_mean = tr_acts[train_labels == 0].mean(dim=0)
        direction = true_mean - false_mean

        # Classify by sign of dot product (centered around midpoint)
        midpoint = (true_mean + false_mean) / 2
        train_preds = ((tr_acts - midpoint) @ direction > 0).float()
        test_preds = ((te_acts - midpoint) @ direction > 0).float()

        train_acc = (train_preds == train_labels).float().mean().item()
        test_acc = (test_preds == test_labels).float().mean().item()
        train_accs.append(train_acc)
        test_accs.append(test_acc)

    return {"train_acc": train_accs, "test_acc": test_accs}


t.manual_seed(42)
all_layers = list(range(NUM_LAYERS))
cities_statements = datasets["cities"]["statement"].tolist()
cities_labels = t.tensor(datasets["cities"]["label"].values, dtype=t.float32)

sweep_results = layer_sweep_accuracy(cities_statements, cities_labels, model, tokenizer, all_layers)

# Print results as a table
sweep_df = pd.DataFrame(
    {
        "Layer": all_layers,
        "Train Acc": [f"{a:.3f}" for a in sweep_results["train_acc"]],
        "Test Acc": [f"{a:.3f}" for a in sweep_results["test_acc"]],
    }
)
display(sweep_df)

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=all_layers, y=sweep_results["train_acc"], mode="lines+markers", name="Train"))
fig.add_trace(go.Scatter(x=all_layers, y=sweep_results["test_acc"], mode="lines+markers", name="Test"))
fig.add_vline(x=PROBE_LAYER, line_dash="dash", line_color="gray", annotation_text=f"Probe layer ({PROBE_LAYER})")
fig.update_layout(
    title="Layer Sweep: Difference-of-Means Accuracy on Cities Dataset",
    xaxis_title="Layer",
    yaxis_title="Accuracy",
    yaxis_range=[0.4, 1.05],
    height=400,
    width=800,
)
fig.show()

best_layer = all_layers[int(np.argmax(sweep_results["test_acc"]))]
print(f"\nBest layer by test accuracy: {best_layer} ({max(sweep_results['test_acc']):.3f})")
print(f"Configured probe layer: {PROBE_LAYER} ({sweep_results['test_acc'][PROBE_LAYER]:.3f})")

,Layer,Train Acc,Test Acc
0,0,0.533,0.480
1,1,0.556,0.537
2,2,0.560,0.540
3,3,0.672,0.657
4,4,0.886,0.850
5,5,0.886,0.857
6,6,0.886,0.850
7,7,0.921,0.900
8,8,0.948,0.943
9,9,0.963,0.937



Best layer by test accuracy: 13 (0.957)
Configured probe layer: 14 (0.947)


# Create Train/Test split

In [17]:
# Create train/test splits for all datasets
t.manual_seed(42)
train_acts, test_acts = {}, {}
train_labels, test_labels = {}, {}

for name in DATASET_NAMES:
    acts = activations[name]
    labs = labels_dict[name]
    n = len(acts)
    perm = t.randperm(n)
    n_train = int(0.8 * n)

    train_acts[name] = acts[perm[:n_train]]
    test_acts[name] = acts[perm[n_train:]]
    train_labels[name] = labs[perm[:n_train]]
    test_labels[name] = labs[perm[n_train:]]

    print(f"{name}: train={n_train}, test={n - n_train}")

cities: train=1196, test=300
sp_en_trans: train=283, test=71
larger_than: train=1584, test=396


# Train a MMProbe

In [18]:
class MMProbe(t.nn.Module):
    def __init__(
        self,
        direction: Float[Tensor, " d_model"],
        covariance: Float[Tensor, "d_model d_model"] | None = None,
        atol: float = 1e-3,
    ):
        super().__init__()
        self.direction = t.nn.Parameter(direction, requires_grad=False)
        if covariance is not None:
            self.inv = t.nn.Parameter(t.linalg.pinv(covariance, hermitian=True, atol=atol), requires_grad=False)
        else:
            self.inv = None

    def forward(self, x: Float[Tensor, "n d_model"], iid: bool = False) -> Float[Tensor, " n"]:
        if iid and self.inv is not None:
            return t.sigmoid(x @ self.inv @ self.direction)
        else:
            return t.sigmoid(x @ self.direction)

    def pred(self, x: Float[Tensor, "n d_model"], iid: bool = False) -> Float[Tensor, " n"]:
        return self(x, iid=iid).round()

    @staticmethod
    def from_data(
        acts: Float[Tensor, "n d_model"],
        labels: Float[Tensor, " n"],
        device: str = "cpu",
    ) -> "MMProbe":
        acts, labels = acts.to(device), labels.to(device)
        pos_acts = acts[labels == 1]
        neg_acts = acts[labels == 0]
        pos_mean = pos_acts.mean(0)
        neg_mean = neg_acts.mean(0)
        direction = pos_mean - neg_mean

        centered = t.cat([pos_acts - pos_mean, neg_acts - neg_mean], dim=0)
        covariance = centered.t() @ centered / acts.shape[0]

        return MMProbe(direction, covariance=covariance).to(device)


mm_probe = MMProbe.from_data(train_acts["cities"], train_labels["cities"])

# Train accuracy
train_preds = mm_probe.pred(train_acts["cities"])
train_acc = (train_preds == train_labels["cities"]).float().mean().item()

# Test accuracy
test_preds = mm_probe.pred(test_acts["cities"])
test_acc = (test_preds == test_labels["cities"]).float().mean().item()
assert test_acc > 0.7, "Expected at least 70% accuracy"

print("MMProbe on cities:")
print(f"  Train accuracy: {train_acc:.3f}")
print(f"  Test accuracy:  {test_acc:.3f}")
print(f"  Direction norm: {mm_probe.direction.norm().item():.3f}")
print(f"  Direction (first 5): {mm_probe.direction[:5].tolist()}")

MMProbe on cities:
  Train accuracy: 0.929
  Test accuracy:  0.900
  Direction norm: 4.847
  Direction (first 5): [0.032529424875974655, -0.07555976510047913, 0.00592588959261775, -0.029578104615211487, 0.013609293848276138]


# Train a LR Probe

In [19]:
class LRProbe(t.nn.Module):
    def __init__(self, d_in: int, scaler_mean: Tensor | None = None, scaler_scale: Tensor | None = None):
        super().__init__()
        self.net = t.nn.Sequential(t.nn.Linear(d_in, 1, bias=False), t.nn.Sigmoid())
        self.register_buffer("scaler_mean", scaler_mean)
        self.register_buffer("scaler_scale", scaler_scale)

    def _normalize(self, x: Float[Tensor, "n d_model"]) -> Float[Tensor, "n d_model"]:
        """Apply StandardScaler normalization if scaler parameters are available."""
        if self.scaler_mean is not None and self.scaler_scale is not None:
            return (x - self.scaler_mean) / self.scaler_scale
        return x

    def forward(self, x: Float[Tensor, "n d_model"]) -> Float[Tensor, " n"]:
        return self.net(self._normalize(x)).squeeze(-1)

    def pred(self, x: Float[Tensor, "n d_model"]) -> Float[Tensor, " n"]:
        return self(x).round()

    @property
    def direction(self) -> Float[Tensor, " d_model"]:
        return self.net[0].weight.data[0]

    @staticmethod
    def from_data(
        acts: Float[Tensor, "n d_model"],
        labels: Float[Tensor, " n"],
        C: float = 0.1,
        device: str = "cpu",
    ) -> "LRProbe":
        """
        Train an LR probe using sklearn's LogisticRegression with StandardScaler normalization.

        Args:
            acts: Activation matrix [n_samples, d_model].
            labels: Binary labels (1=true, 0=false).
            C: Inverse regularization strength (lower = stronger regularization).
                Default 0.1 (reg_coeff=10) matches the deception-detection paper's cfg.yaml.
                The repo class default is reg_coeff=1000 (C=0.001), which is stronger.
            device: Device to place the resulting probe on.
        """
        X = acts.cpu().float().numpy()
        y = labels.cpu().float().numpy()

        # Standardize features (zero mean, unit variance) before fitting, as in the paper
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # fit_intercept=False: the paper fits on normalized data so the intercept is redundant
        lr_model = LogisticRegression(C=C, random_state=42, fit_intercept=False, max_iter=1000)
        lr_model.fit(X_scaled, y)

        # Build probe with scaler parameters baked in
        scaler_mean = t.tensor(scaler.mean_, dtype=t.float32)
        scaler_scale = t.tensor(scaler.scale_, dtype=t.float32)
        probe = LRProbe(acts.shape[-1], scaler_mean=scaler_mean, scaler_scale=scaler_scale).to(device)
        probe.net[0].weight.data[0] = t.tensor(lr_model.coef_[0], dtype=t.float32).to(device)

        return probe


lr_probe = LRProbe.from_data(train_acts["cities"], train_labels["cities"], device="cpu")

# Train accuracy
train_preds = lr_probe.pred(train_acts["cities"])
train_acc = (train_preds == train_labels["cities"]).float().mean().item()

# Test accuracy
test_preds = lr_probe.pred(test_acts["cities"])
test_acc = (test_preds == test_labels["cities"]).float().mean().item()

print("LRProbe on cities:")
print(f"  Train accuracy: {train_acc:.3f}")
print(f"  Test accuracy:  {test_acc:.3f}")
print(f"  Direction norm: {lr_probe.direction.norm().item():.3f}")
assert test_acc >= 0.90, f"Test accuracy too low: {test_acc:.3f} (expected >= 0.90)"

# Compare directions
mm_dir = mm_probe.direction / mm_probe.direction.norm()
lr_dir = lr_probe.direction / lr_probe.direction.norm()
cos_sim = (mm_dir @ lr_dir).item()
print(f"\nCosine similarity between MM and LR directions: {cos_sim:.4f}")

# Compare both probes across all 3 datasets
results_rows = []
for name in DATASET_NAMES:
    mm_p = MMProbe.from_data(train_acts[name], train_labels[name])
    lr_p = LRProbe.from_data(train_acts[name], train_labels[name])

    mm_test_acc = (mm_p.pred(test_acts[name]) == test_labels[name]).float().mean().item()
    lr_test_acc = (lr_p.pred(test_acts[name]) == test_labels[name]).float().mean().item()
    results_rows.append({"Dataset": name, "MM Test Acc": f"{mm_test_acc:.3f}", "LR Test Acc": f"{lr_test_acc:.3f}"})

results_df = pd.DataFrame(results_rows)
print("\nProbe accuracy comparison across datasets:")
display(results_df)

# Bar chart
fig = go.Figure()
fig.add_trace(go.Bar(name="MMProbe", x=DATASET_NAMES, y=[float(r["MM Test Acc"]) for r in results_rows]))
fig.add_trace(go.Bar(name="LRProbe", x=DATASET_NAMES, y=[float(r["LR Test Acc"]) for r in results_rows]))
fig.update_layout(
    title="Probe Test Accuracy by Dataset",
    yaxis_title="Test Accuracy",
    yaxis_range=[0.5, 1.05],
    barmode="group",
    height=400,
    width=600,
)
fig.show()

LRProbe on cities:
  Train accuracy: 1.000
  Test accuracy:  1.000
  Direction norm: 0.439

Cosine similarity between MM and LR directions: 0.6345

Probe accuracy comparison across datasets:


,Dataset,MM Test Acc,LR Test Acc
0,cities,0.900,1.000
1,sp_en_trans,0.986,1.000
2,larger_than,0.995,1.000


# Train a DCT Probe

In [ ]:
class DCTProbe(t.nn.Module):
    """
    Fully unsupervised probe: the direction comes entirely from DCT + judge,
    no truth labels used. We combine top-k DCT vectors (weighted by their
    judge truthfulness delta) into a single probe direction.
    """
    def __init__(self, direction: Float[Tensor, " d_model"]):
        super().__init__()
        self.direction = t.nn.Parameter(direction, requires_grad=False)

    def forward(self, x: Float[Tensor, "n d_model"]) -> Float[Tensor, " n"]:
        return t.sigmoid(x @ self.direction)

    def pred(self, x: Float[Tensor, "n d_model"]) -> Float[Tensor, " n"]:
        return self(x).round()

    @staticmethod
    def from_vectors(
        V: Float[Tensor, "d_model num_factors"],
        mean_deltas: pd.Series,
        k: int,
        device: str = "cpu",
    ) -> "DCTProbe":
        top_indices = mean_deltas.nlargest(k).index.tolist()
        top_deltas = t.tensor(
            [mean_deltas[i] for i in top_indices], dtype=t.float32
        )

        # Weight each DCT vector by its judge delta
        top_vectors = V[:, top_indices].to(dtype=t.float32)  # [d_model, k]
        direction = top_vectors @ top_deltas  # [d_model]
        direction = direction / direction.norm()

        return DCTProbe(direction).to(device)

NameError: name 't' is not defined

In [ ]:
# Load DCT vectors and judge results
data = t.load("dct_probes/vectors/dct_vectors.pt", weights_only=True)
V = data["V"]

judge_df = pd.read_json("dct_probes/results/judge_results.jsonl", lines=True)

# Compute per-factor truthfulness delta
baseline = judge_df[judge_df["factor_idx"] == -1].set_index("prompt_id")["judge_score"]
steered = judge_df[judge_df["factor_idx"] >= 0].copy()
steered = steered.merge(baseline.rename("baseline_score"), on="prompt_id")
steered["delta"] = steered["baseline_score"] - steered["judge_score"]
mean_deltas = steered.groupby("factor_idx")["delta"].mean().sort_values(ascending=False)

# Select top-k truth-relevant directions
TOP_K = 10
top_indices = mean_deltas.nlargest(TOP_K).index.tolist()
print(f"Top {TOP_K} factors by truth-reduction: {top_indices}")
print(f"Their deltas: {[f'{mean_deltas[i]:.2f}' for i in top_indices]}")

In [ ]:
# Single dataset test
dct_probe = DCTProbe.from_data(
    train_acts["cities"], train_labels["cities"],
    V=V, top_indices=top_indices,
)

train_preds = dct_probe.pred(train_acts["cities"])
train_acc = (train_preds == train_labels["cities"]).float().mean().item()
test_preds = dct_probe.pred(test_acts["cities"])
test_acc = (test_preds == test_labels["cities"]).float().mean().item()

print(f"DCTProbe (k={TOP_K}) on cities:")
print(f"  Train accuracy: {train_acc:.3f}")
print(f"  Test accuracy:  {test_acc:.3f}")

# Compare directions with MM and LR
dct_dir = dct_probe.direction / dct_probe.direction.norm()
mm_dir = mm_probe.direction / mm_probe.direction.norm()
lr_dir = lr_probe.direction / lr_probe.direction.norm()
print(f"  Cosine sim with MM: {(dct_dir @ mm_dir).item():.4f}")
print(f"  Cosine sim with LR: {(dct_dir @ lr_dir).item():.4f}")

In [ ]:
# Wrapper to make DCTProbe work with compute_generalization_matrix
class DCTProbeFactory:
    """Wraps DCTProbe so it has the same from_data(acts, labels) signature."""
    def __init__(self, V, top_indices):
        self.V = V
        self.top_indices = top_indices

    def from_data(self, acts, labels, device="cpu"):
        return DCTProbe.from_data(
            acts, labels, V=self.V, top_indices=self.top_indices, device=device
        )

dct_factory = DCTProbeFactory(V, top_indices)
dct_matrix = compute_generalization_matrix(
    train_acts, train_labels, test_acts, test_labels,
    DATASET_NAMES, dct_factory,
)

# Test subspace directions

In [ ]:
def k_sweep_accuracy(
    train_acts: Float[Tensor, "n d_model"],
    train_labels: Float[Tensor, " n"],
    test_acts: Float[Tensor, "n d_model"],
    test_labels: Float[Tensor, " n"],
    V: Float[Tensor, "d_model num_factors"],
    mean_deltas: pd.Series,
    k_values: list[int],
) -> dict[str, list[float]]:
    """
    For each k, select the top-k DCT vectors by judge delta,
    build a DCTProbe, and compute train/test accuracy.

    Args:
        train_acts, train_labels: Training activations and labels at the probe layer.
        test_acts, test_labels: Test activations and labels at the probe layer.
        V: DCT steering vectors [d_model, num_factors].
        mean_deltas: Per-factor mean truthfulness delta from judge (higher = more truth-reducing).
        k_values: List of k values to sweep over.

    Returns:
        Dict with keys "train_acc" and "test_acc", each a list of accuracies per k.
    """
    train_accs = []
    test_accs = []

    for k in k_values:
        top_indices = mean_deltas.nlargest(k).index.tolist()
        probe = DCTProbe.from_data(
            train_acts, train_labels, V=V, top_indices=top_indices
        )

        train_preds = probe.pred(train_acts)
        test_preds = probe.pred(test_acts)

        train_acc = (train_preds == train_labels).float().mean().item()
        test_acc = (test_preds == test_labels).float().mean().item()

        train_accs.append(train_acc)
        test_accs.append(test_acc)

    return {"train_acc": train_accs, "test_acc": test_accs}


# Run the sweep
k_values = [1, 2, 3, 5, 8, 10, 15, 20, 30, 50]
# Filter to valid range
k_values = [k for k in k_values if k <= V.shape[1]]

sweep_results = k_sweep_accuracy(
    train_acts["cities"], train_labels["cities"],
    test_acts["cities"], test_labels["cities"],
    V, mean_deltas, k_values,
)

# Print results
sweep_df = pd.DataFrame({
    "k": k_values,
    "Train Acc": [f"{a:.3f}" for a in sweep_results["train_acc"]],
    "Test Acc": [f"{a:.3f}" for a in sweep_results["test_acc"]],
})
display(sweep_df)

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=k_values, y=sweep_results["train_acc"],
    mode="lines+markers", name="Train",
))
fig.add_trace(go.Scatter(
    x=k_values, y=sweep_results["test_acc"],
    mode="lines+markers", name="Test",
))

# Add MM and LR baselines as horizontal lines
mm_test_acc = (mm_probe.pred(test_acts["cities"]) == test_labels["cities"]).float().mean().item()
lr_test_acc = (lr_probe.pred(test_acts["cities"]) == test_labels["cities"]).float().mean().item()

fig.add_hline(y=mm_test_acc, line_dash="dash", line_color="red",
              annotation_text=f"MM baseline ({mm_test_acc:.3f})")
fig.add_hline(y=lr_test_acc, line_dash="dash", line_color="blue",
              annotation_text=f"LR baseline ({lr_test_acc:.3f})")

fig.update_layout(
    title="DCT Probe Accuracy vs Number of Steering Vectors (k)",
    xaxis_title="k (number of DCT directions)",
    yaxis_title="Accuracy",
    yaxis_range=[0.4, 1.05],
    height=400,
    width=800,
)
fig.show()

best_k = k_values[int(np.argmax(sweep_results["test_acc"]))]
print(f"\nBest k by test accuracy: {best_k} ({max(sweep_results['test_acc']):.3f})")
print(f"MM baseline test accuracy: {mm_test_acc:.3f}")
print(f"LR baseline test accuracy: {lr_test_acc:.3f}")

# Cross-dataset generalization

In [20]:
def compute_generalization_matrix(
    train_acts: dict[str, Float[Tensor, "n d"]],
    train_labels: dict[str, Float[Tensor, " n"]],
    test_acts: dict[str, Float[Tensor, "n d"]],
    test_labels: dict[str, Float[Tensor, " n"]],
    dataset_names: list[str],
    probe_cls: type,
) -> Float[Tensor, "n_datasets n_datasets"]:
    """
    Compute a generalization matrix: entry (i, j) is the test accuracy of a probe trained on dataset i
    and evaluated on dataset j.

    Args:
        train_acts, train_labels: Training data per dataset.
        test_acts, test_labels: Test data per dataset.
        dataset_names: Names of datasets (determines matrix ordering).
        probe_cls: Probe class to use (MMProbe or LRProbe), must have from_data and pred methods.

    Returns:
        Tensor of shape [n_datasets, n_datasets] with accuracy values.
    """
    n = len(dataset_names)
    matrix = t.zeros(n, n)
    for i, train_name in enumerate(dataset_names):
        probe = probe_cls.from_data(train_acts[train_name], train_labels[train_name])
        for j, test_name in enumerate(dataset_names):
            preds = probe.pred(test_acts[test_name])
            acc = (preds == test_labels[test_name]).float().mean().item()
            matrix[i, j] = acc
    return matrix


mm_matrix = compute_generalization_matrix(train_acts, train_labels, test_acts, test_labels, DATASET_NAMES, MMProbe)
lr_matrix = compute_generalization_matrix(train_acts, train_labels, test_acts, test_labels, DATASET_NAMES, LRProbe)

assert mm_matrix.shape == (3, 3), f"Wrong shape: {mm_matrix.shape}"
assert (mm_matrix.diag() > 0.6).all(), "In-distribution accuracy should be at least 60%"

# Heatmap visualization
fig = make_subplots(rows=1, cols=2, subplot_titles=["MMProbe", "LRProbe"], horizontal_spacing=0.15)

for idx, (matrix, name) in enumerate([(mm_matrix, "MM"), (lr_matrix, "LR")]):
    text_vals = [[f"{matrix[i, j]:.3f}" for j in range(len(DATASET_NAMES))] for i in range(len(DATASET_NAMES))]
    fig.add_trace(
        go.Heatmap(
            z=matrix.numpy(),
            x=DATASET_NAMES,
            y=DATASET_NAMES,
            text=text_vals,
            texttemplate="%{text}",
            colorscale="RdYlGn",
            zmin=0.5,
            zmax=1.0,
            showscale=(idx == 1),
        ),
        row=1,
        col=idx + 1,
    )
    fig.update_yaxes(title_text="Train dataset" if idx == 0 else "", row=1, col=idx + 1)
    fig.update_xaxes(title_text="Test dataset", row=1, col=idx + 1)

fig.update_layout(title="Cross-dataset Generalization (Test Accuracy)", height=400, width=800)
fig.show()

# Cosine similarity between probe directions
mm_directions = {name: MMProbe.from_data(train_acts[name], train_labels[name]).direction for name in DATASET_NAMES}
lr_directions = {name: LRProbe.from_data(train_acts[name], train_labels[name]).direction for name in DATASET_NAMES}

print("\nPairwise cosine similarity between probe directions:")
for probe_name, directions in [("MM", mm_directions), ("LR", lr_directions)]:
    print(f"\n  {probe_name}Probe:")
    for i, n1 in enumerate(DATASET_NAMES):
        for j, n2 in enumerate(DATASET_NAMES):
            if j > i:
                d1 = directions[n1] / directions[n1].norm()
                d2 = directions[n2] / directions[n2].norm()
                print(f"    {n1} vs {n2}: {(d1 @ d2).item():.4f}")


Pairwise cosine similarity between probe directions:

  MMProbe:
    cities vs sp_en_trans: 0.3443
    cities vs larger_than: 0.3112
    sp_en_trans vs larger_than: 0.2274

  LRProbe:
    cities vs sp_en_trans: 0.2445
    cities vs larger_than: 0.1331
    sp_en_trans vs larger_than: 0.1792


# Examples of Steering